<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/08-window-functions/02-lag-lead-first-last.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)

print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))


# 8.2 — lag, lead, first, last

Rows looking at other rows: previous/next (lag/lead), and the ends
of the window (first/last) — including the last_value frame trap and
the ignorenulls gap-fill pattern.

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("8.2-lag-lead-first-last")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

orders = (
    spark.read.csv(f"{DATA_DIR}/orders.csv", header=True, inferSchema=True)
    .withColumn("revenue", F.round(col("quantity") * col("unit_price"), 2))
)

# unique order within customer: date, then order_id (the 8.1 discipline)
w = Window.partitionBy("customer_id").orderBy("order_date", "order_id")

## lag and lead

First row of each partition: no predecessor -> null (meaningful!) or
your default. Offsets and defaults are positional args.

In [ ]:
(orders.select(
    "customer_id", "order_id", "order_date", "revenue",
    F.lag("revenue").over(w).alias("prev_rev"),
    F.lag("revenue", 1, 0.0).over(w).alias("prev_or_0"),
    F.lead("order_date").over(w).alias("next_date"),
 )
 .filter(col("customer_id").isin("c001", "c002", "c003"))
 .orderBy("customer_id", "order_date", "order_id")
 .show())

## The canonical use: deltas and gaps

Days between consecutive orders — the self-join that 7.3 needed
aliases for, as one expression. Change-vs-previous works the same.

In [ ]:
gaps = (orders
        .withColumn("prev_date", F.lag("order_date").over(w))
        .withColumn("days_since_prev", F.datediff("order_date", F.lag("order_date").over(w)))
        .withColumn("rev_change", F.round(col("revenue") - F.lag("revenue").over(w), 2)))

(gaps.filter(col("customer_id").isin("c001", "c004", "c012"))
 .select("customer_id", "order_id", "order_date", "days_since_prev", "revenue", "rev_change")
 .orderBy("customer_id", "order_date", "order_id")
 .show())

In [ ]:
# aggregate the gaps: average days between orders, per customer
(gaps.filter(col("days_since_prev").isNotNull())
 .groupBy("customer_id")
 .agg(F.round(F.avg("days_since_prev"), 1).alias("avg_gap_days"),
      F.count("*").alias("n_gaps"))
 .orderBy("avg_gap_days")
 .show(5))

## The last_value trap

Default frame with orderBy ends AT THE CURRENT ROW — so last() just
returns the current row's value. first() works only by accident.

In [ ]:
w_ordered = Window.partitionBy("customer_id").orderBy("order_date", "order_id")
w_full = w_ordered.rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)

(orders.select(
    "customer_id", "order_id", "product",
    F.first("product").over(w_ordered).alias("first_ok"),      # partition start — fine
    F.last("product").over(w_ordered).alias("last_TRAP"),      # == current row!
    F.last("product").over(w_full).alias("last_fixed"),        # explicit full frame
 )
 .filter(col("customer_id") == "c001")
 .show(truncate=False))

In [ ]:
# production-preferred fix: first() over DESCENDING unique order
w_desc = Window.partitionBy("customer_id").orderBy(F.desc("order_date"), F.desc("order_id"))
(orders
 .withColumn("latest_product", F.first("product").over(w_desc))
 .filter(col("customer_id") == "c001")
 .select("order_id", "order_date", "product", "latest_product")
 .show(truncate=False))

## ignorenulls: gap-fill / last-observation-carried-forward

LOCF's honest habitat: readings where "missing means unchanged"
(sensors, balances, slowly-changing attributes). Frame spelled out —
after the trap above, never leave last() on an implicit frame.

In [ ]:
readings = spark.createDataFrame(
    [("s1", 1, 20.0), ("s1", 2, None), ("s1", 3, None), ("s1", 4, 22.5),
     ("s1", 5, None), ("s2", 1, None), ("s2", 2, 18.0), ("s2", 3, None)],
    "sensor STRING, minute INT, temp DOUBLE",
)

w_locf = (Window.partitionBy("sensor")
          .orderBy("minute")
          .rowsBetween(Window.unboundedPreceding, Window.currentRow))

(readings
 .withColumn("temp_filled", F.last("temp", ignorenulls=True).over(w_locf))
 .orderBy("sensor", "minute")
 .show())
# s1 minutes 2,3 inherit 20.0; minute 5 inherits 22.5. s2 minute 1 is a
# LEADING null — nothing precedes it, stays null (exercise 2 adds the
# fallback). Note what LOCF asserts about the world: missing == unchanged.
# The null quantities in orders.csv do NOT qualify (they're data-quality
# holes, and both are their customer's first order anyway) — fillna per
# 3.5 stays the honest fix there.

## Exercises

1. `pct_change`: revenue change vs previous order as a percentage,
   null-safe (previous revenue 0 or null must not divide-by-zero —
   when/otherwise from 3.2).
2. Fix s2's leading null: extend LOCF with a fallback chain — carried
   value, else the NEXT non-null (a backward fill: first(ignorenulls)
   over the mirrored frame), else the sensor's mean. F.coalesce
   composes all three.
3. "Days until next order" with lead, then flag each order
   `churn_risk = days_until_next > 10 OR days_until_next IS NULL`.
   Which of those two conditions is a business assumption in disguise?
4. nth_value: each order annotated with its customer's SECOND order
   date (frame matters — think before you run). Cross-check against
   a row_number()==2 approach.
5. SQL round trip: rewrite the gaps query with LAG(...) OVER in
   spark.sql(), including `LEAD(order_date) IGNORE NULLS` on a
   nullable column to see syntax PySpark's lead() can't express.

In [ ]:
# your exercise code here